# 🧠 Brain Tumor Detection — MONAI Pipeline
> **AI Engineer Notebook** | Framework: MONAI | Classes: Glioma · Meningioma · Pituitary · No Tumor

---
| Section | Description |
|---------|-------------|
| 1 | Environment & Imports |
| 2 | Dataset Download (Kaggle) |
| 3 | MONAI Transforms & DataLoader |
| 4 | Model Architecture (DenseNet121) |
| 5 | Training Loop |
| 6 | Evaluation & Metrics |
| 7 | Inference |


In [ ]:
# ─────────────────────────────────────────────
# SECTION 1 — Install & Import
# ─────────────────────────────────────────────
!pip install monai[all] kaggle --quiet

import os, json, shutil, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from monai.networks.nets import DenseNet121
from monai.transforms import (
    Compose, LoadImage, EnsureChannelFirst, ScaleIntensity,
    Resize, RandRotate90, RandFlip, RandZoom,
    RandAdjustContrast, ToTensor, NormalizeIntensity
)
from monai.data import Dataset, DataLoader
from monai.metrics import ROCAUCMetric
from monai.utils import set_determinism
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

set_determinism(seed=42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2 — Download Dataset (Kaggle)
# ─────────────────────────────────────────────
import os
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri --unzip -p brain_tumor_data

DATA_ROOT = Path('brain_tumor_data')
CLASSES   = ['glioma', 'meningioma', 'notumor', 'pituitary']
TRAIN_DIR = DATA_ROOT / 'Training'
TEST_DIR  = DATA_ROOT / 'Testing'
print('✅ Dataset ready')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 3 — EDA: Class Distribution
# ─────────────────────────────────────────────
counts = {c: len(list((TRAIN_DIR / c).glob('*'))) for c in CLASSES}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(counts.keys(), counts.values(), color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[0].set_title('Train Class Distribution'); axes[0].set_ylabel('Count')

# Sample images
axes[1].axis('off')
fig2, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, cls in zip(axs, CLASSES):
    img_path = list((TRAIN_DIR / cls).glob('*'))[0]
    ax.imshow(Image.open(img_path), cmap='gray')
    ax.set_title(cls.upper()); ax.axis('off')
plt.suptitle('Sample MRI per Class', fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# SECTION 4 — MONAI Data Pipeline
# ─────────────────────────────────────────────
IMG_SIZE = 224
BATCH    = 32

def build_file_list(root_dir):
    data = []
    for label, cls in enumerate(CLASSES):
        for p in (root_dir / cls).glob('*'):
            data.append({'image': str(p), 'label': label})
    random.shuffle(data)
    return data

train_files = build_file_list(TRAIN_DIR)
test_files  = build_file_list(TEST_DIR)

# Split 90/10 from training
split = int(len(train_files) * 0.9)
val_files   = train_files[split:]
train_files = train_files[:split]

print(f'Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}')

In [ ]:
# ─────────────────────────────────────────────
# MONAI Transforms
# ─────────────────────────────────────────────
train_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize(spatial_size=(IMG_SIZE, IMG_SIZE)),
    ScaleIntensity(),
    NormalizeIntensity(nonzero=True),
    RandRotate90(prob=0.5),
    RandFlip(prob=0.5, spatial_axis=1),
    RandZoom(prob=0.3, min_zoom=0.9, max_zoom=1.1),
    RandAdjustContrast(prob=0.3),
    ToTensor(),
])

val_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize(spatial_size=(IMG_SIZE, IMG_SIZE)),
    ScaleIntensity(),
    NormalizeIntensity(nonzero=True),
    ToTensor(),
])

class TumorDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img  = self.transform(item['image'])
        # Ensure 3 channels (RGB)
        if img.shape[0] == 1:
            img = img.repeat(3, 1, 1)
        elif img.shape[0] == 4:
            img = img[:3]
        return img, torch.tensor(item['label'], dtype=torch.long)

train_ds = TumorDataset(train_files, train_transforms)
val_ds   = TumorDataset(val_files,   val_transforms)
test_ds  = TumorDataset(test_files,  val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

print('✅ DataLoaders ready')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 5 — Model: MONAI DenseNet121
# ─────────────────────────────────────────────
model = DenseNet121(
    spatial_dims=2,
    in_channels=3,
    out_channels=len(CLASSES),
    pretrained=True
).to(DEVICE)

criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer  = Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler  = CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

print('✅ MONAI DenseNet121 initialized')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 6 — Training Loop
# ─────────────────────────────────────────────
EPOCHS     = 30
best_val   = 0.0
history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    # — Train —
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * imgs.size(0)
        t_correct += (logits.argmax(1) == labels).sum().item()
        t_total   += imgs.size(0)

    # — Validate —
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += (logits.argmax(1) == labels).sum().item()
            v_total   += imgs.size(0)

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(v_loss / v_total)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    scheduler.step()

    if v_acc > best_val:
        best_val = v_acc
        torch.save(model.state_dict(), 'best_tumor_monai.pth')

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {t_loss/t_total:.4f} Acc: {t_acc:.4f} | '
          f'Val Loss: {v_loss/v_total:.4f} Acc: {v_acc:.4f}')

print(f'\n🏆 Best Val Accuracy: {best_val:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 7 — Training Curves
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', color='#4C72B0')
axes[0].plot(history['val_loss'],   label='Val',   color='#DD8452')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(history['train_acc'], label='Train', color='#55A868')
axes[1].plot(history['val_acc'],   label='Val',   color='#C44E52')
axes[1].set_title('Accuracy Curve'); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.suptitle('MONAI DenseNet121 — Training History', fontsize=14)
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# SECTION 8 — Evaluation on Test Set
# ─────────────────────────────────────────────
model.load_state_dict(torch.load('best_tumor_monai.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(DEVICE)
        logits = model(imgs)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('📊 Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASSES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix — MONAI DenseNet121')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# SECTION 9 — Grad-CAM Visualization
# ─────────────────────────────────────────────
!pip install grad-cam --quiet
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target last DenseBlock
target_layers = [model.features.denseblock4]
cam = GradCAM(model=model, target_layers=target_layers)

sample_imgs, sample_labels = next(iter(test_loader))
fig, axs = plt.subplots(2, 4, figsize=(18, 8))

for i in range(4):
    img_t  = sample_imgs[i:i+1].to(DEVICE)
    label  = sample_labels[i].item()
    target = [ClassifierOutputTarget(label)]
    mask   = cam(input_tensor=img_t, targets=target)[0]

    img_np = sample_imgs[i].permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    overlay = show_cam_on_image(img_np.astype(np.float32), mask, use_rgb=True)

    axs[0, i].imshow(img_np); axs[0, i].set_title(f'Original: {CLASSES[label]}'); axs[0, i].axis('off')
    axs[1, i].imshow(overlay); axs[1, i].set_title('Grad-CAM'); axs[1, i].axis('off')

plt.suptitle('Grad-CAM — Model Attention Visualization', fontsize=14)
plt.tight_layout(); plt.savefig('gradcam.png', dpi=150); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# SECTION 10 — Inference (Single Image)
# ─────────────────────────────────────────────
def predict(image_path: str) -> dict:
    import torch.nn.functional as F
    model.eval()
    img = val_transforms(image_path)
    if img.shape[0] == 1: img = img.repeat(3, 1, 1)
    img = img.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(img)
        probs  = F.softmax(logits, dim=1)[0].cpu().numpy()
    pred_idx  = probs.argmax()
    return {
        'prediction': CLASSES[pred_idx],
        'confidence': float(probs[pred_idx]),
        'all_probs': dict(zip(CLASSES, probs.tolist()))
    }

# Example usage
sample_path = str(list((TEST_DIR / 'glioma').glob('*'))[0])
result = predict(sample_path)

print(f'🔍 Prediction: {result["prediction"].upper()}')
print(f'   Confidence: {result["confidence"]*100:.2f}%')
print(f'\n   Class Probabilities:')
for cls, prob in result['all_probs'].items():
    bar = '█' * int(prob * 30)
    print(f'   {cls:12s} {bar} {prob*100:.2f}%')

img_show = Image.open(sample_path)
plt.figure(figsize=(5, 5))
plt.imshow(img_show, cmap='gray')
plt.title(f'Pred: {result["prediction"]} ({result["confidence"]*100:.1f}%)')
plt.axis('off'); plt.tight_layout(); plt.show()